# NLP FINAL TEAM PROJECT

**A Chatbot For the Government of Ghana**

What this notebook contains:

1. Environment Setup
2. Data Source Configuration
3. Web Scraping (PDFs + HTML)
4. Text Extraction & Processing
5. Chunking for RAG
6. Persistence with Google Drive


In [1]:
# Install required packages
!pip install -q PyMuPDF selenium webdriver-manager beautifulsoup4 requests

# Setup Chrome for Selenium (Colab specific)
!apt-get update > /dev/null 2>&1
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add - > /dev/null 2>&1
!echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list
!apt-get update > /dev/null 2>&1
!apt-get install -y google-chrome-stable > /dev/null 2>&1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 96.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.0/512.0 kB 32.5 MB/s eta 0:00:00


In [2]:
from google.colab import drive
import os
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

# Create project directory
PROJECT_DIR = Path('/content/drive/MyDrive/GhanaGovRAG')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


In [ ]:
!pip install PyMuPDF
!pip install requests
!pip install beautifulsoup4


In [11]:
import time
import json
import requests
import fitz  # PyMuPDF
import hashlib
import re
from datetime import datetime
from typing import List, Dict, Optional
from urllib.parse import urljoin, urlparse
from bs4 import BeautifulSoup
import glob

# Selenium imports
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager




In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [12]:
pdf_path = glob.glob("/content/drive/MyDrive/GhanaGovRAG/*.pdf")

In [4]:
print("CONFIGURATION")


class Config:
    """Central configuration for the pipeline"""

    # Directories (using Google Drive for persistence)
    BASE_DIR = PROJECT_DIR
    RAW_PDF_DIR = BASE_DIR / "raw_pdfs"
    PROCESSED_DIR = BASE_DIR / "processed"
    METADATA_DIR = BASE_DIR / "metadata"
    LOGS_DIR = BASE_DIR / "logs"

    # Processing parameters
    CHUNK_SIZE = 1000  # characters per chunk
    CHUNK_OVERLAP = 200  # overlap between chunks

    # Scraping parameters
    REQUEST_DELAY = 2  # seconds between requests
    MAX_RETRIES = 3

    @classmethod
    def setup_directories(cls):
        """Create all necessary directories"""
        for dir_path in [cls.RAW_PDF_DIR, cls.PROCESSED_DIR,
                         cls.METADATA_DIR, cls.LOGS_DIR]:
            dir_path.mkdir(parents=True, exist_ok=True)
        print(f" Directories have been created in: {cls.BASE_DIR}")

# Setup directories
Config.setup_directories()

CONFIGURATION
 Directories have been created in: /content/drive/MyDrive/GhanaGovRAG


In [6]:
print("DATA SOURCE REGISTRY")

class DataSourceRegistry:
    """
    Registry for all data sources.
    Supports:
        - Web scraping (Selenium/HTML/PDF)
        - Local PDF uploads
    """

    SOURCES = {
        "bank_of_ghana": {
            "name": "Bank of Ghana",
            "base_url": "https://www.bog.gov.gh",
            "pages": [
                {
                    "url": "https://www.bog.gov.gh/publications/annual-report/",
                    "category": "Annual Report",
                    "type": "selenium",
                    "content_type": "pdf"
                },
                {
                    "url": "https://www.bog.gov.gh/publications/quarterly-bulletin/",
                    "category": "Quarterly Bulletin",
                    "type": "selenium",
                    "content_type": "pdf"
                },
                {
                    "url": "https://www.bog.gov.gh/publications/news-briefs/",
                    "category": "News Briefs",
                    "type": "selenium",
                    "content_type": "pdf"
                },
                {
                    "url": "https://www.bog.gov.gh/publications/staff-working-papers/",
                    "category": "Staff Working Papers",
                    "type": "selenium",
                    "content_type": "pdf"
                },

            ]
        },

        # local uploads source
        "local_uploads": {
            "name": "Local Uploaded Files",
            "base_url": None,
            "pages": [
                # Example entry:
                # {
                #     "path": "/content/my_uploaded_file.pdf",
                #     "category": "Manual Uploads",
                #     "type": "local",
                #     "content_type": "pdf"
                # }
            ]
        }
    }

    @classmethod
    def add_source(cls, source_id: str, source_config: dict):
        """Dynamically add a new data source."""
        cls.SOURCES[source_id] = source_config
        print(f"Added new source: {source_config['name']}")

    @classmethod
    def add_local_pdf(cls, file_path: str, category: str = "Manual Uploads"):
        """helper function to register a local PDF stored in Colab or PC upload."""
        entry = {
            "path": file_path,
            "category": category,
            "type": "local",
            "content_type": "pdf"
        }

        cls.SOURCES["local_uploads"]["pages"].append(entry)
        print(f"Registered local PDF: {file_path}")

    @classmethod
    def get_all_sources(cls):
        """Return all registered sources."""
        return cls.SOURCES

    @classmethod
    def get_source(cls, source_id: str):
        """Get a specific data source by ID."""
        return cls.SOURCES.get(source_id)


print("Data sources registered")


DATA SOURCE REGISTRY
 Data sources registered


In [7]:
print("MULTI-FORMAT SCRAPER")

class MultiFormatScraper:
    """
    Handles scraping PDFs and HTML content from government websites
    """

    def __init__(self, driver=None):
        self.driver = driver
        self.download_log = []

    def scrape_html_content(self, url: str) -> Optional[Dict]:
        """Scrape and extract text content directly from HTML pages"""
        print(f"  Scraping HTML: {url}")

        try:
            if self.driver:
                self.driver.get(url)
                time.sleep(2)
                html_content = self.driver.page_source
            else:
                response = requests.get(url, timeout=30)
                response.raise_for_status()
                html_content = response.text

            soup = BeautifulSoup(html_content, 'html.parser')

            # Remove unwanted elements
            for script in soup(["script", "style", "nav", "footer", "header"]):
                script.decompose()

            # Get title
            title = soup.find('title')
            title = title.get_text().strip() if title else "Untitled"

            # Try to find main content
            main_content = None
            for selector in ['main', 'article', '.content', '#content', '.main-content']:
                main_content = soup.select_one(selector)
                if main_content:
                    break

            if not main_content:
                main_content = soup.find('body')

            # Extract text
            if main_content:
                text = main_content.get_text(separator='\n', strip=True)
                text = re.sub(r'\n\s*\n', '\n\n', text)
            else:
                text = soup.get_text(separator='\n', strip=True)

            return {
                "success": True,
                "url": url,
                "title": title,
                "text": text,
                "word_count": len(text.split()),
                "type": "html"
            }

        except Exception as e:
            print(f"    ✗ Failed: {str(e)}")
            return {"success": False, "url": url, "error": str(e)}

    def scrape_with_selenium(self, url: str) -> List[str]:
        """Extract PDF links using Selenium"""
        if not self.driver:
            raise ValueError("Selenium driver not initialized")

        print(f"  Scanning page: {url}")
        self.driver.get(url)
        time.sleep(3)

        pdf_links = set()

        # Find all anchor tags
        a_tags = self.driver.find_elements("tag name", "a")
        for a in a_tags:
            href = a.get_attribute("href")
            if href and href.lower().endswith(".pdf"):
                pdf_links.add(href)

        # Check buttons
        buttons = self.driver.find_elements("tag name", "button")
        for btn in buttons:
            pdf_url = btn.get_attribute("data-url") or btn.get_attribute("href")
            if pdf_url and pdf_url.lower().endswith(".pdf"):
                pdf_links.add(pdf_url)

        print(f"    Found {len(pdf_links)} PDFs")
        return list(pdf_links)

    def download_pdf(self, url: str, save_dir: Path, filename: Optional[str] = None) -> Optional[str]:
        """Download a single PDF with retry logic"""
        if not filename:
            filename = url.split("/")[-1]
            filename = re.sub(r'[^\w\-_\. ]', '_', filename)

        filepath = save_dir / filename

        # Skip if exists
        if filepath.exists():
            print(f"  Skipped (exists): {filename}")
            return str(filepath)

        # Download with retries
        for attempt in range(Config.MAX_RETRIES):
            try:
                response = requests.get(url, stream=True, timeout=30)
                response.raise_for_status()

                with open(filepath, 'wb') as f:
                    for chunk in response.iter_content(chunk_size=8192):
                        f.write(chunk)

                print(f"    ✓ Downloaded: {filename}")
                self.download_log.append({
                    "url": url,
                    "filename": filename,
                    "status": "success",
                    "timestamp": datetime.now().isoformat()
                })
                return str(filepath)

            except Exception as e:
                if attempt < Config.MAX_RETRIES - 1:
                    print(f"    ⚠️  Retry {attempt + 1}/{Config.MAX_RETRIES}")
                    time.sleep(2)
                else:
                    print(f"    ✗ Failed: {filename}")
                    self.download_log.append({
                        "url": url,
                        "filename": filename,
                        "status": "failed",
                        "error": str(e),
                        "timestamp": datetime.now().isoformat()
                    })
                    return None
        return None

    def scrape_source(self, source_config: dict, save_dir: Path) -> List[Dict]:
        """Scrape all PDFs from a source"""
        all_pdfs = []

        for page in source_config["pages"]:
            url = page["url"]
            category = page["category"]
            scrape_type = page["type"]
            content_type = page.get("content_type", "pdf")

            if content_type != "pdf":
                continue  # Skip non-PDF for this method

            # Get PDF links
            if scrape_type == "selenium":
                pdf_links = self.scrape_with_selenium(url)

            # Download PDFs
            for pdf_url in pdf_links:
                time.sleep(Config.REQUEST_DELAY)
                filepath = self.download_pdf(pdf_url, save_dir)

                if filepath:
                    all_pdfs.append({
                        "source": source_config["name"],
                        "category": category,
                        "url": pdf_url,
                        "filepath": filepath,
                        "page_url": url,
                        "type": "pdf"
                    })

        return all_pdfs

    def scrape_html_pages_source(self, source_config: dict, save_dir: Path) -> List[Dict]:
        """Scrape HTML content from a source"""
        all_html = []

        for page in source_config["pages"]:
            url = page["url"]
            category = page["category"]
            content_type = page.get("content_type", "pdf")

            if content_type != "html":
                continue  # Skip non-HTML

            time.sleep(Config.REQUEST_DELAY)
            html_content = self.scrape_html_content(url)

            if html_content and html_content["success"]:
                filename = hashlib.md5(url.encode()).hexdigest()[:12] + ".json"
                filepath = save_dir / filename

                with open(filepath, 'w', encoding='utf-8') as f:
                    json.dump(html_content, f, indent=2, ensure_ascii=False)

                all_html.append({
                    "source": source_config["name"],
                    "category": category,
                    "url": url,
                    "filepath": str(filepath),
                    "type": "html",
                    "title": html_content["title"]
                })

        return all_html

print("Multi-format scraper ready")

MULTI-FORMAT SCRAPER
Multi-format scraper ready
